In [75]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [76]:
df  = pd.DataFrame()

## Sentiment

In [77]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()
df_time  = pd.DataFrame()
df_hour  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl', compression=None)
df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
df_sentiment["sentiment"] = df_sentiment["sentiment"].ffill()

df_sentiment["sentiment_yesterday"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment_day_before_yesterday"] = df_sentiment["sentiment"].shift(periods=2)

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment_yesterday"]-df_sentiment["sentiment_day_before_yesterday"])/df_sentiment["sentiment_day_before_yesterday"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment_yesterday"]/df_sentiment["sentiment_day_before_yesterday"])
# SENTIMENT EWM
df_sentiment["sentiment_yesterday_ewm"]=df_sentiment["sentiment_yesterday"].ewm(10).mean()

In [78]:
df = df_sentiment

In [79]:
df

,sentiment,Date,PressReleaseCount,sentiment_yesterday,sentiment_day_before_yesterday,sentiment_return,sentiment_ln_return,sentiment_yesterday_ewm
2019-09-25,94.117647,2019-09-25,13,NaN,NaN,NaN,NaN,NaN
2019-09-26,85.826772,2019-09-26,11,94.117647,NaN,NaN,NaN,94.117647
2019-09-27,94.545455,2019-09-27,7,85.826772,94.117647,-0.088091,-0.092215,89.774808
2019-09-30,90.000000,2019-09-30,10,94.545455,85.826772,0.101585,0.096750,91.518760
2019-10-01,100.000000,2019-10-01,12,90.000000,94.545455,-0.048077,-0.049271,91.083192
...,...,...,...,...,...,...,...,...
2021-08-19,71.779141,2021-08-19,25,68.047337,81.595092,-0.166036,-0.181566,78.180475
2021-08-20,74.545455,2021-08-20,13,71.779141,68.047337,0.054841,0.053390,77.598536
2021-08-23,74.698795,2021-08-23,29,74.545455,71.779141,0.038539,0.037815,77.320983
2021-08-24,85.326087,2021-08-24,30,74.698795,74.545455,0.002057,0.002055,77.082602


## VIX

In [81]:
from yahoo_fin.stock_info import get_data

df_vix  = pd.DataFrame()
df_vix = get_data("^VIX", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
# vix LN RETURN
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [82]:
df = pd.merge(df_vix,df,how='outer', on='Date')

In [83]:
df

,Date,vix,vix_yesterday,vix_return,vix_ln_return,vix_ewm,sentiment,PressReleaseCount,sentiment_yesterday,sentiment_day_before_yesterday,sentiment_return,sentiment_ln_return,sentiment_yesterday_ewm
0,2019-09-25,15.960000,NaN,NaN,NaN,15.960000,94.117647,13.0,NaN,NaN,NaN,NaN,NaN
1,2019-09-26,16.070000,15.960000,0.006892,0.006869,16.017619,85.826772,11.0,94.117647,NaN,NaN,NaN,94.117647
2,2019-09-27,17.219999,16.070000,0.071562,0.069117,16.457160,94.545455,7.0,85.826772,94.117647,-0.088091,-0.092215,89.774808
3,2019-09-30,16.240000,17.219999,-0.056911,-0.058594,16.394880,90.000000,10.0,94.545455,85.826772,0.101585,0.096750,91.518760
4,2019-10-01,18.559999,16.240000,0.142857,0.133531,16.914110,100.000000,12.0,90.000000,94.545455,-0.048077,-0.049271,91.083192
...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,2021-08-19,21.670000,21.570000,0.004636,0.004625,17.871938,71.779141,25.0,68.047337,81.595092,-0.166036,-0.181566,78.180475
480,2021-08-20,18.559999,21.670000,-0.143516,-0.154920,17.934489,74.545455,13.0,71.779141,68.047337,0.054841,0.053390,77.598536
481,2021-08-23,17.150000,18.559999,-0.075970,-0.079011,17.863172,74.698795,29.0,74.545455,71.779141,0.038539,0.037815,77.320983
482,2021-08-24,17.219999,17.150000,0.004082,0.004073,17.804702,85.326087,30.0,74.698795,74.545455,0.002057,0.002055,77.082602


## S&P500

In [84]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")

In [88]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [89]:
df = pd.merge(df_sp500,df,how='outer', on='Date')

In [97]:
# df = df.drop([475])

,Date,sp500,sp500_yesterday,sp500_return,sp500_ln_return,sp500_ewm,vix,vix_yesterday,vix_return,vix_ln_return,vix_ewm,sentiment,PressReleaseCount,sentiment_yesterday,sentiment_day_before_yesterday,sentiment_return,sentiment_ln_return,sentiment_yesterday_ewm
454,2021-07-15,4360.029785,4374.299805,-0.003262,-0.003268,4313.753877,17.010000,16.330000,0.041641,0.040798,16.749783,91.891892,38.0,90.384615,90.909091,-0.005769,-0.005786,88.049293
455,2021-07-16,4327.160156,4360.029785,-0.007539,-0.007567,4314.972630,18.450001,17.010000,0.084656,0.081263,16.904348,73.228346,15.0,91.891892,90.384615,0.016676,0.016539,88.398620
456,2021-07-19,4258.490234,4327.160156,-0.015870,-0.015997,4309.837867,22.500000,18.450001,0.219512,0.198451,17.413044,98.159509,25.0,73.228346,91.891892,-0.203103,-0.227030,87.019504
457,2021-07-20,4323.060059,4258.490234,0.015163,0.015049,4311.039884,19.730000,22.500000,-0.123111,-0.131375,17.623676,85.057471,24.0,98.159509,73.228346,0.340458,0.293011,88.032232
458,2021-07-21,4358.689941,4323.060059,0.008242,0.008208,4315.371707,17.910000,19.730000,-0.092245,-0.096781,17.649706,84.276730,26.0,85.057471,98.159509,-0.133477,-0.143267,87.761799
459,2021-07-22,4367.479980,4358.689941,0.002017,0.002015,4320.108823,17.690001,17.910000,-0.012284,-0.012360,17.653369,89.565217,26.0,84.276730,85.057471,-0.009179,-0.009221,87.444974
460,2021-07-23,4411.790039,4367.479980,0.010145,0.010094,4328.443479,17.200001,17.690001,-0.027699,-0.028090,17.612154,79.885057,8.0,89.565217,84.276730,0.062751,0.060861,87.637724
461,2021-07-26,4422.299805,4411.790039,0.002382,0.002379,4336.975872,17.580000,17.200001,0.022093,0.021852,17.609230,87.323944,20.0,79.885057,89.565217,-0.108079,-0.114378,86.932936
462,2021-07-27,4401.459961,4422.299805,-0.004712,-0.004724,4342.838062,19.360001,17.580000,0.101251,0.096447,17.768391,86.554622,33.0,87.323944,79.885057,0.093120,0.089036,86.968482
463,2021-07-28,4400.640137,4401.459961,-0.000186,-0.000186,4348.092796,18.309999,19.360001,-0.054236,-0.055762,17.817629,91.240876,34.0,86.554622,87.323944,-0.008810,-0.008849,86.930858


# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# MERGE

Missing Values, how to handle

Option 1: Fill NaN with Outlier or Zero
In this specific example filling the missing value with an outlier value such as np.inf or 0 seems to be very naive. However, using values like -999, is sometimes a good idea.

Option 2: Fill NaN with Mean Value
Filling NaNs with the mean value is also not sufficient and naive, and doesn't seems to be a good option.

Option 3: Fill NaN with Last Value with .ffill()
Filling NaNs with the last value could be bit better.

Option 4: Fill NaN with Linearly ffilld Value with .ffill()
Filling NaNs with the ffilld values is the best option in this small examlple but it requires knowledge of the neighouring value

# THREE PERIODS

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-08-25']

In [ ]:
# df_part3 = df_part3.ffill()

In [ ]:
# df.reset_index(drop=True, inplace=True)
# df_part1.reset_index(drop=True, inplace=True)
# df_part2.reset_index(drop=True, inplace=True)
# df_part3.reset_index(drop=True, inplace=True)

In [ ]:
# df.sort_index()

# DF HOURLY

In [ ]:
df_time=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl', compression=None)
df_hour=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesDateHour.pkl', compression=None)

In [ ]:
df_hour.Date= pd.to_datetime(df_hour.Date)

In [ ]:
df_hour = df_hour.merge(df_sp500, on='Date', how='left')

In [ ]:
sp500_yesterday = SP500["sp500"].shift(periods=1)
SP500["sp500_ln_return"]=np.log(SP500["sp500"]/sp500_yesterday)

In [ ]:
df_hour = df_hour.merge(SP500, on='Date', how='left')

In [ ]:
# Prepare data
df_hour['year'] = [d.year for d in df_hour.Date]
df_hour['month'] = [d.strftime('%b') for d in df_hour.Date]
df_hour['day'] = [d.day_name() for d in df_hour.Date]
years = df_hour['year'].unique()
df_hour['month-year'] = [d.strftime('%b') + str(d.year) for d in df_hour.Date]

In [ ]:
df_hour.index = df_hour.Date

In [ ]:
df_hour.sort_values(by="Hour")

In [ ]:
df_hour_part1 = df_hour.loc['2019-09-25':'2020-02-14']
df_hour_part2 = df_hour.loc['2020-02-15':'2020-05-31']
df_hour_part3 = df_hour.loc['2020-06-01':'2021-08-25']

In [ ]:
df_hour["sentiment"]

# 09-16

In [ ]:
df_09_15=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_09_15.pkl', compression=None)
df_16_08=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_16_08.pkl', compression=None)
df_09_15.index= pd.to_datetime(df_09_15.index)
df_16_08.index= pd.to_datetime(df_16_08.index)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
# Prepare data df_09_15
df_09_15['year'] = [d.year for d in df_09_15.Date]
df_09_15['month'] = [d.strftime('%b') for d in df_09_15.Date]
df_09_15['day'] = [d.day_name() for d in df_09_15.Date]
years = df_09_15['year'].unique()
df_09_15['month-year'] = [d.strftime('%b') + str(d.year) for d in df_09_15.Date]

# Prepare data df_16_08
df_16_08['year'] = [d.year for d in df_16_08.Date]
df_16_08['month'] = [d.strftime('%b') for d in df_16_08.Date]
df_16_08['day'] = [d.day_name() for d in df_16_08.Date]
years = df_16_08['year'].unique()
df_16_08['month-year'] = [d.strftime('%b') + str(d.year) for d in df_16_08.Date]


In [ ]:
df_09_15 = df_09_15.merge(df_vix,on='Date',how='left')
df_09_15 = df_09_15.merge(SP500, on='Date', how='left')

df_16_08 = df_16_08.merge(df_vix,on='Date',how='left')
df_16_08 = df_16_08.merge(SP500, on='Date', how='left')

In [ ]:
df_09_15.index= pd.to_datetime(df_09_15.Date)
df_16_08.index= pd.to_datetime(df_16_08.Date)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].apply(np.float)
df_16_08["sentiment"] = df_16_08["sentiment"].apply(np.float)

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_09_15["sentiment_ln_return"]=np.log(df_09_15["sentiment"]/sentiment_yesterday)


sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_16_08["sentiment_ln_return"]=np.log(df_16_08["sentiment"]/sentiment_yesterday)

In [ ]:
df_09_15["sentiment"] = df_09_15["sentiment"].ffill()
df_16_08["sentiment"] = df_16_08["sentiment"].ffill()

In [ ]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
df_09_15["sentiment_return"]=((df_09_15["sentiment"]-sentiment_yesterday)/sentiment_yesterday)
sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
df_16_08["sentiment_return"]=((df_16_08["sentiment"]-sentiment_yesterday)/sentiment_yesterday)

In [ ]:
df_09_15["sentiment_return"] = df_09_15["sentiment_return"].ffill()
df_16_08["sentiment_return"] = df_16_08["sentiment_return"].ffill()

In [ ]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text